# INTEX EDA + ML Pipeline

This notebook:
- Loads and profiles all CSVs in this folder
- Builds a resident-level modeling table by aggregating related tables
- Trains a reproducible scikit-learn pipeline to predict a target (default: `current_risk_level`)

> Tip: If you want a different target, change `TARGET_COL` below (e.g., `case_status`, `reintegration_status`).

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

DATA_DIR = Path.cwd()  # assumes you opened this notebook from the same folder as the CSVs
DATA_DIR

PosixPath('/Users/masonzarges/Desktop/INTEX Data')

In [2]:
# Load all CSVs in the folder into a dict of DataFrames
csv_paths = sorted(DATA_DIR.glob("*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}")

dfs: dict[str, pd.DataFrame] = {}
for p in csv_paths:
    key = p.stem
    dfs[key] = pd.read_csv(p)

list(dfs.keys()), {k: v.shape for k, v in dfs.items()}

(['home_visitations',
  'in_kind_donation_items',
  'incident_reports',
  'intervention_plans',
  'partner_assignments',
  'partners',
  'process_recordings',
  'public_impact_snapshots',
  'residents',
  'safehouse_monthly_metrics',
  'safehouses',
  'social_media_posts',
  'supporters'],
 {'home_visitations': (1337, 14),
  'in_kind_donation_items': (129, 9),
  'incident_reports': (100, 12),
  'intervention_plans': (180, 11),
  'partner_assignments': (48, 9),
  'partners': (30, 12),
  'process_recordings': (2819, 15),
  'public_impact_snapshots': (50, 7),
  'residents': (60, 49),
  'safehouse_monthly_metrics': (450, 11),
  'safehouses': (9, 13),
  'social_media_posts': (812, 39),
  'supporters': (60, 15)})

In [3]:
def profile_df(df: pd.DataFrame, name: str, n: int = 5) -> None:
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    }).sort_values(["missing", "n_unique"], ascending=[False, False]).head(30))
    print(f"\n{name}: shape={df.shape}")
    display(df.head(n))

for name, df in dfs.items():
    print("=" * 90)
    profile_df(df, name)

,dtype,missing,missing_%,n_unique
follow_up_notes,object,549,41.06,1
family_members_present,object,425,31.79,100
visitation_id,int64,0,0.00,1337
visit_date,object,0,0.00,783
resident_id,int64,0,0.00,58
social_worker,object,0,0.00,20
location_visited,object,0,0.00,6
visit_type,object,0,0.00,5
purpose,object,0,0.00,5
observations,object,0,0.00,5



home_visitations: shape=(1337, 14)


,visitation_id,resident_id,visit_date,social_worker,visit_type,location_visited,family_members_present,purpose,observations,family_cooperation_level,safety_concerns_noted,follow_up_needed,follow_up_notes,visit_outcome
0,1,1,2023-11-02,SW-04,Routine Follow-Up,Proposed Foster Home,Lopez (Parent); Diaz (Sibling),Visitation for routine follow-up,Visit observations recorded during routine fol...,Neutral,True,False,Follow-up scheduled,Favorable
1,2,1,2023-11-22,SW-12,Routine Follow-Up,Church,Mendoza (Parent); Mendoza (Sibling),Visitation for routine follow-up,Visit observations recorded during routine fol...,Neutral,False,False,NaN,Favorable
2,3,1,2023-12-14,SW-16,Post-Placement Monitoring,Barangay Office,Santos (Parent); Torres (Sibling),Visitation for post-placement monitoring,Visit observations recorded during post-placem...,Uncooperative,False,True,Follow-up scheduled,Unfavorable
3,4,1,2023-12-18,SW-07,Reintegration Assessment,Proposed Foster Home,Cruz (Parent); Mendoza (Sibling),Visitation for reintegration assessment,Visit observations recorded during reintegrati...,Cooperative,False,True,Follow-up scheduled,Needs Improvement
4,5,1,2023-12-24,SW-05,Post-Placement Monitoring,Community Center,NaN,Visitation for post-placement monitoring,Visit observations recorded during post-placem...,Cooperative,False,True,NaN,Unfavorable


,dtype,missing,missing_%,n_unique
item_id,int64,0,0.0,129
estimated_unit_value,float64,0,0.0,129
donation_id,int64,0,0.0,98
quantity,int64,0,0.0,28
item_name,object,0,0.0,9
item_category,object,0,0.0,7
unit_of_measure,object,0,0.0,5
intended_use,object,0,0.0,5
received_condition,object,0,0.0,3



in_kind_donation_items: shape=(129, 9)


...


social_media_posts: shape=(812, 39)


,post_id,platform,platform_post_id,post_url,created_at,day_of_week,post_hour,post_type,media_type,caption,hashtags,num_hashtags,mentions_count,has_call_to_action,call_to_action_type,content_topic,sentiment_tone,caption_length,features_resident_story,campaign_name,is_boosted,boost_budget_php,impressions,reach,likes,comments,shares,saves,click_throughs,video_views,engagement_rate,profile_visits,donation_referrals,estimated_donation_value_php,follower_count_at_post,watch_time_seconds,avg_view_duration_seconds,subscriber_count_at_post,forwards
0,318,WhatsApp,wa_4293211912553134,https://whatsapp.com/channel/lighthouse_ph/429...,2023-01-05 18:52:00,Thursday,18,FundraisingAppeal,Text,"This is hard to ask, but our reserve is gone. ...",NaN,0,3,True,LearnMore,Education,Grateful,157,False,NaN,False,NaN,1580,1093,118,36,22,9,48,NaN,0.1105,21,10,21473.25,1522,NaN,NaN,NaN,50.0
1,529,Instagram,ig_5129900136072862,https://instagram.com/p/sYhZp-0AvhH,2023-01-06 11:30:00,Friday,11,EducationalContent,Photo,What does freedom mean to a trafficking surviv...,"#SurvivorStrong, #BeTheChange, #HumanTrafficki...",4,0,False,NaN,Education,Celebratory,150,False,NaN,False,NaN,6362,4395,548,110,149,59,85,NaN,0.1745,335,2,4708.45,1833,NaN,NaN,NaN,NaN
2,86,LinkedIn,li_2326736034499294,https://linkedin.com/feed/update/urn:li:activi...,2023-01-08 10:14:00,Sunday,10,EventPromotion,Text,SAVE THE DATE! Join us on January 21 for Fundr...,NaN,0,0,False,NaN,Reintegration,Urgent,138,False,NaN,False,NaN,554,336,27,7,12,4,3,NaN,0.1411,8,0,0.00,457,NaN,NaN,NaN,NaN
3,380,Instagram,ig_4154485528046983,https://instagram.com/p/1LSXA225Jpv,2023-01-09 15:06:00,Monday,15,ThankYou,Video,Every donation is a prayer answered. Thank you...,"#ProtectChildren, #BeTheChange",2,1,False,NaN,Education,Emotional,113,False,NaN,False,NaN,4309,2948,190,55,45,16,35,3313.0,0.0677,62,0,0.00,1796,NaN,NaN,NaN,NaN
4,425,TikTok,tk_7166643297225195,https://tiktok.com/@lighthouse_ph/video/817153...,2023-01-09 15:59:00,Monday,15,ThankYou,Reel,Big thanks to Juan for the recent donation. Yo...,"#SurvivorStrong, #EndViolence, #HumanTrafficki...",4,1,True,LearnMore,Education,Hopeful,129,False,NaN,True,4030.64,23175,14008,728,232,141,79,474,17974.0,0.0802,172,2,8351.49,916,NaN,NaN,NaN,NaN


,dtype,missing,missing_%,n_unique
organization_name,object,56,93.33,4
first_name,object,4,6.67,56
last_name,object,4,6.67,55
first_donation_date,object,1,1.67,58
supporter_id,int64,0,0.00,60
display_name,object,0,0.00,60
email,object,0,0.00,60
phone,object,0,0.00,60
created_at,object,0,0.00,60
supporter_type,object,0,0.00,6



supporters: shape=(60, 15)


,supporter_id,supporter_type,display_name,organization_name,first_name,last_name,relationship_type,region,country,email,phone,status,created_at,first_donation_date,acquisition_channel
0,1,SocialMediaAdvocate,Mila Alvarez,NaN,Mila,Alvarez,Local,Luzon,Philippines,mila-alvarez@smart.com.ph,+63 997 578 1887,Active,2022-01-01 00:00:00,2023-07-02,SocialMedia
1,2,Volunteer,Aria Brown,NaN,Aria,Brown,Local,Mindanao,Philippines,aria-brown@pldt.net.ph,+63 927 354 4139,Active,2022-01-06 00:00:00,2023-09-25,SocialMedia
2,3,MonetaryDonor,Noah Chen,NaN,Noah,Chen,Local,Luzon,Philippines,noah-chen@globe.com.ph,+63 917 553 2604,Active,2022-01-11 00:00:00,2023-06-25,SocialMedia
3,4,MonetaryDonor,Liam Diaz,NaN,Liam,Diaz,PartnerOrganization,Mindanao,Philippines,liam-diaz@globe.com.ph,+63 945 516 8956,Active,2022-01-16 00:00:00,2026-03-01,Church
4,5,InKindDonor,Emma Evans,NaN,Emma,Evans,PartnerOrganization,Mindanao,Philippines,emma-evans@yahoo.com.ph,+63 995 371 8454,Active,2022-01-21 00:00:00,2024-01-18,Website


## Build a resident-level modeling dataset

We’ll use `residents` as the main table and left-join/aggregate related tables by `resident_id`.

Default prediction target: `residents.current_risk_level` (multiclass classification).

In [4]:
TARGET_COL = "current_risk_level"  # change if desired
ID_COL = "resident_id"

residents = dfs["residents"].copy()

# Basic cleanup: normalize obvious boolean-like columns to actual bool where possible
bool_like_cols = [c for c in residents.columns if c.startswith("sub_cat_") or c.startswith("family_") or c in {"is_pwd", "has_special_needs"}]
for c in bool_like_cols:
    if c in residents.columns:
        residents[c] = residents[c].map({True: True, False: False, "True": True, "False": False}).astype("boolean")

# Parse key dates (errors coerced to NaT)
for c in [
    "date_of_birth",
    "date_of_admission",
    "date_enrolled",
    "date_closed",
    "created_at",
]:
    if c in residents.columns:
        residents[c] = pd.to_datetime(residents[c], errors="coerce")

residents[[ID_COL, TARGET_COL]].head()

,resident_id,current_risk_level
0,1,High
1,2,Medium
2,3,Medium
3,4,Low
4,5,Low


In [5]:
def agg_incidents(inc: pd.DataFrame) -> pd.DataFrame:
    inc = inc.copy()
    if "incident_date" in inc.columns:
        inc["incident_date"] = pd.to_datetime(inc["incident_date"], errors="coerce")

    out = inc.groupby(ID_COL).agg(
        incident_count=("incident_id", "count"),
        incident_high_sev=("severity", lambda s: (s.astype(str).str.lower() == "high").sum()),
        incident_unresolved=("resolved", lambda s: (~s.fillna(False).astype(bool)).sum()),
        incident_types_n=("incident_type", pd.Series.nunique),
        last_incident_date=("incident_date", "max"),
    )
    out["days_since_last_incident"] = (
        (pd.Timestamp.today().normalize() - out["last_incident_date"]).dt.days
    )
    return out.drop(columns=["last_incident_date"])


def agg_visits(vis: pd.DataFrame) -> pd.DataFrame:
    vis = vis.copy()
    if "visit_date" in vis.columns:
        vis["visit_date"] = pd.to_datetime(vis["visit_date"], errors="coerce")

    out = vis.groupby(ID_COL).agg(
        home_visit_count=("visitation_id", "count"),
        home_visit_types_n=("visit_type", pd.Series.nunique),
        home_visit_outcomes_n=("visit_outcome", pd.Series.nunique),
        safety_concerns_count=("safety_concerns_noted", lambda s: s.fillna(False).astype(bool).sum()),
        follow_up_needed_count=("follow_up_needed", lambda s: s.fillna(False).astype(bool).sum()),
        last_visit_date=("visit_date", "max"),
    )
    out["days_since_last_visit"] = (
        (pd.Timestamp.today().normalize() - out["last_visit_date"]).dt.days
    )
    return out.drop(columns=["last_visit_date"])


def agg_plans(plans: pd.DataFrame) -> pd.DataFrame:
    plans = plans.copy()
    for c in ["target_date", "case_conference_date", "created_at", "updated_at"]:
        if c in plans.columns:
            plans[c] = pd.to_datetime(plans[c], errors="coerce")

    out = plans.groupby(ID_COL).agg(
        intervention_plan_count=("plan_id", "count"),
        intervention_categories_n=("plan_category", pd.Series.nunique),
        intervention_status_n=("status", pd.Series.nunique),
        intervention_target_value_mean=("target_value", "mean"),
        last_plan_update=("updated_at", "max"),
    )
    out["days_since_last_plan_update"] = (
        (pd.Timestamp.today().normalize() - out["last_plan_update"]).dt.days
    )
    return out.drop(columns=["last_plan_update"])


def agg_sessions(proc: pd.DataFrame) -> pd.DataFrame:
    proc = proc.copy()
    if "session_date" in proc.columns:
        proc["session_date"] = pd.to_datetime(proc["session_date"], errors="coerce")

    out = proc.groupby(ID_COL).agg(
        session_count=("recording_id", "count"),
        session_minutes_sum=("session_duration_minutes", "sum"),
        session_types_n=("session_type", pd.Series.nunique),
        concerns_flagged_count=("concerns_flagged", lambda s: s.fillna(False).astype(bool).sum()),
        referral_made_count=("referral_made", lambda s: s.fillna(False).astype(bool).sum()),
        last_session_date=("session_date", "max"),
    )
    out["days_since_last_session"] = (
        (pd.Timestamp.today().normalize() - out["last_session_date"]).dt.days
    )
    return out.drop(columns=["last_session_date"])


features = residents.copy()

if "incident_reports" in dfs:
    features = features.merge(agg_incidents(dfs["incident_reports"]), how="left", left_on=ID_COL, right_index=True)
if "home_visitations" in dfs:
    features = features.merge(agg_visits(dfs["home_visitations"]), how="left", left_on=ID_COL, right_index=True)
if "intervention_plans" in dfs:
    features = features.merge(agg_plans(dfs["intervention_plans"]), how="left", left_on=ID_COL, right_index=True)
if "process_recordings" in dfs:
    features = features.merge(agg_sessions(dfs["process_recordings"]), how="left", left_on=ID_COL, right_index=True)

features.shape, features.columns[:15]

((60, 71),
 Index(['resident_id', 'case_control_no', 'internal_code', 'safehouse_id', 'case_status', 'sex', 'date_of_birth', 'birth_status',
        'place_of_birth', 'religion', 'case_category', 'sub_cat_orphaned', 'sub_cat_trafficked', 'sub_cat_child_labor',
        'sub_cat_physical_abuse'],
       dtype='object'))

In [6]:
# Prepare train table

df_model = features.copy()

# Drop columns that are identifiers or free-text notes (usually not stable / can leak)
drop_cols = [
    "case_control_no",
    "internal_code",
    "referring_agency_person",
    "notes_restricted",
]
df_model = df_model.drop(columns=[c for c in drop_cols if c in df_model.columns])

# Target + filters
if TARGET_COL not in df_model.columns:
    raise KeyError(f"TARGET_COL={TARGET_COL!r} not found in residents columns")

# Keep rows with a known target
mask = df_model[TARGET_COL].notna() & (df_model[TARGET_COL].astype(str).str.len() > 0)
df_train = df_model.loc[mask].copy()

# Basic feature engineering from dates
if "date_of_birth" in df_train.columns:
    df_train["age_years"] = ((pd.Timestamp.today().normalize() - df_train["date_of_birth"]).dt.days / 365.25).round(2)

if "date_of_admission" in df_train.columns:
    df_train["days_since_admission"] = (pd.Timestamp.today().normalize() - df_train["date_of_admission"]).dt.days

# Drop raw datetime columns (keep engineered numeric features instead)
datetime_cols = df_train.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
df_train = df_train.drop(columns=datetime_cols)

# Separate X/y
y = df_train[TARGET_COL].astype(str)
X = df_train.drop(columns=[TARGET_COL])

X.shape, y.value_counts(dropna=False).head(20)

((60, 63),
 current_risk_level
 Low         34
 Medium      20
 High         5
 Critical     1
 Name: count, dtype: int64)

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Identify column types
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in numeric_cols]

# Ensure categorical columns are plain python objects (avoids edge-cases with pandas string dtypes)
X = X.copy()
for c in cat_cols:
    X[c] = X[c].astype("object")

# Preprocessing
numeric_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_tf, numeric_cols),
        ("cat", categorical_tf, cat_cols),
    ],
    remainder="drop",
)

# Model
clf = LogisticRegression(max_iter=3000)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", clf),
])

# Robust split: only stratify when every class has >= 2 rows
class_counts = y.value_counts()
use_stratify = class_counts.min() >= 2
stratify_arg = y if use_stratify else None
if not use_stratify:
    print(
        "Warning: some classes have <2 rows; disabling stratified split.\n"
        f"Class counts: {class_counts.to_dict()}"
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_arg,
)

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

Class counts: {'Low': 34, 'Medium': 20, 'High': 5, 'Critical': 1}
              precision    recall  f1-score   support

    Critical       0.00      0.00      0.00         0
        High       0.00      0.00      0.00         1
         Low       0.80      0.50      0.62         8
      Medium       0.33      0.67      0.44         3

    accuracy                           0.50        12
   macro avg       0.28      0.29      0.26        12
weighted avg       0.62      0.50      0.52        12



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
# Train on all labeled rows and generate predictions for all residents
pipe.fit(X, y)

# Build prediction frame for all residents (including those with missing target)
df_pred = df_model.copy()

# Recreate the same engineered date features used for training
if "date_of_birth" in df_pred.columns:
    dob = pd.to_datetime(df_pred["date_of_birth"], errors="coerce")
    df_pred["age_years"] = ((pd.Timestamp.today().normalize() - dob).dt.days / 365.25).round(2)

if "date_of_admission" in df_pred.columns:
    doa = pd.to_datetime(df_pred["date_of_admission"], errors="coerce")
    df_pred["days_since_admission"] = (pd.Timestamp.today().normalize() - doa).dt.days

# Drop datetime columns (pipeline expects same column types as training)
datetime_cols = df_pred.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
df_pred = df_pred.drop(columns=datetime_cols)

X_all = df_pred.drop(columns=[TARGET_COL]) if TARGET_COL in df_pred.columns else df_pred

pred_label = pipe.predict(X_all)

pred_out = pd.DataFrame({
    ID_COL: df_pred[ID_COL],
    f"pred_{TARGET_COL}": pred_label,
})

# Optional: add probabilities (for multiclass, one column per class)
if hasattr(pipe.named_steps["model"], "predict_proba"):
    proba = pipe.predict_proba(X_all)
    classes = pipe.named_steps["model"].classes_
    for i, cls in enumerate(classes):
        pred_out[f"proba_{cls}"] = proba[:, i]

pred_out.head()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,resident_id,pred_current_risk_level,proba_Critical,proba_High,proba_Low,proba_Medium
0,1,High,5.347441e-17,9.999695e-01,1.122748e-07,0.000030
1,2,Medium,1.532674e-106,7.478916e-10,2.792344e-02,0.972077
2,3,Medium,8.872400e-83,2.969671e-05,1.402563e-01,0.859714
3,4,Low,7.663693e-12,3.245910e-02,8.934149e-01,0.074126
4,5,Low,1.960180e-100,1.831850e-03,9.291689e-01,0.068999


In [9]:
# Save predictions
out_path = DATA_DIR / f"predictions_{TARGET_COL}.csv"
pred_out.to_csv(out_path, index=False)
out_path

PosixPath('/Users/masonzarges/Desktop/INTEX Data/predictions_current_risk_level.csv')